In [1]:
!pip install harmonypy=='0.0.9'

In [2]:
!pip install anndata=='0.8.0'

In [5]:
from samalg import SAM
import scanpy as sc
import statistics
import matplotlib.pyplot as plt
import seaborn as sns
import random
import pandas as pd
import matplotlib.colors
import scipy
import numpy as np
import sklearn.metrics as metrics
from scipy import sparse
import anndata as ad
from sklearn.manifold import TSNE

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#input: combined h5ad of data
#output: SAM object

In [6]:
fn = '../../Testing_Raw_Dat_RNASEQ_Joined/tot_dat_DR_ncbi_joined_cleaned_07172026.h5ad'

In [7]:
dataframe = sc.read_h5ad(fn)

KeyboardInterrupt: 

In [ ]:
#if rerunning on something that has already been normalized in the past
dataframe.X = dataframe.obsm['Raw_X']

In [6]:
for item in dataframe.X[0,:]:
    print(item)

  (0, 6)	22.0
  (0, 5238)	3.0
  (0, 5237)	3.0
  (0, 5)	10.0
  (0, 1)	18.0
  (0, 4)	20.0
  (0, 3)	4.0
  (0, 5236)	9.0
  (0, 5235)	2.0
  (0, 14666)	2.0
  (0, 14665)	1.0
  (0, 8663)	1.0
  (0, 12194)	1.0
  (0, 5375)	1.0
  (0, 6205)	1.0
  (0, 18421)	2.0
  (0, 21713)	1.0
  (0, 10595)	7.0
  (0, 8928)	1.0
  (0, 11164)	1.0
  (0, 23023)	1.0
  (0, 15265)	1.0
  (0, 24638)	1.0
  (0, 10446)	1.0
  (0, 14726)	1.0
  :	:
  (0, 23436)	1.0
  (0, 11323)	11.0
  (0, 17367)	1.0
  (0, 17191)	1.0
  (0, 23946)	1.0
  (0, 8623)	1.0
  (0, 15231)	1.0
  (0, 12835)	1.0
  (0, 14069)	1.0
  (0, 15486)	1.0
  (0, 7346)	1.0
  (0, 14027)	1.0
  (0, 19714)	1.0
  (0, 18492)	1.0
  (0, 13684)	1.0
  (0, 21430)	1.0
  (0, 20587)	1.0
  (0, 23661)	3.0
  (0, 24072)	1.0
  (0, 13707)	1.0
  (0, 18438)	1.0
  (0, 23598)	2.0
  (0, 4495)	1.0
  (0, 23241)	1.0
  (0, 18413)	3.0


In [ ]:
#Make sure gene names are in BLAST Table

In [7]:
mapping = pd.read_csv('../../BLASTMAPPING/maps/active_maps/hypo_proj/drmg/mg_to_dr.txt', delimiter = '\t', header = None)

In [8]:
gene_set = set(mapping[1])

In [9]:
a = 0
for item in dataframe.var_names:
    if item in gene_set:
        a += 1
a 

21541

In [ ]:
#Check whether the data is "raw" (UMI counts)

In [10]:
print(dataframe.X[10,:])

  (0, 6)	3.0
  (0, 5238)	2.0
  (0, 5237)	1.0
  (0, 1)	3.0
  (0, 4)	17.0
  (0, 3)	2.0
  (0, 5236)	2.0
  (0, 5235)	1.0
  (0, 14666)	2.0
  (0, 18396)	1.0
  (0, 12076)	1.0
  (0, 13315)	1.0
  (0, 18180)	1.0
  (0, 19074)	1.0
  (0, 10446)	1.0
  (0, 15045)	1.0
  (0, 8786)	1.0
  (0, 521)	1.0
  (0, 18416)	3.0
  (0, 6909)	1.0
  (0, 5473)	1.0
  (0, 22535)	1.0
  (0, 23698)	1.0
  (0, 14074)	1.0
  (0, 6787)	1.0
  :	:
  (0, 9678)	1.0
  (0, 11295)	1.0
  (0, 24433)	1.0
  (0, 24246)	1.0
  (0, 17512)	4.0
  (0, 25218)	1.0
  (0, 24656)	1.0
  (0, 8406)	1.0
  (0, 17192)	2.0
  (0, 15189)	1.0
  (0, 14271)	1.0
  (0, 21613)	1.0
  (0, 9094)	1.0
  (0, 18492)	4.0
  (0, 8090)	1.0
  (0, 23661)	1.0
  (0, 22846)	1.0
  (0, 13707)	1.0
  (0, 23256)	3.0
  (0, 25428)	1.0
  (0, 8566)	1.0
  (0, 11177)	1.0
  (0, 8842)	2.0
  (0, 13790)	1.0
  (0, 18413)	5.0


In [ ]:
#Check number of counts/genes

In [11]:
statistics.median(dataframe.obs['n_genes'])

800.0

In [ ]:
#plotting n_genes

In [ ]:
from matplotlib import rcParams

plt.rcParams['figure.figsize'] = 20,10
sns.reset_orig()
sns.violinplot(x = 'key', y = 'n_genes', data = dataframe.obs, color = 'white')
sns.stripplot(x = 'key', y = 'n_genes', data = dataframe.obs, color = 'black', size = 1)
plt.xticks(rotation=45, ha = 'right')
plt.ylim((0,6000))
plt.xlabel(None)
plt.tight_layout()
plt.savefig(fn + '_ngenes.png')
plt.show()

In [ ]:
#plotting n counts

In [ ]:
from matplotlib import rcParams

rcParams['figure.figsize'] = 20,10
sns.reset_orig()
sns.violinplot(x = 'key', y = 'n_counts', data = dataframe.obs, color = 'white')
sns.stripplot(x = 'key', y = 'n_counts', data = dataframe.obs, color = 'black', size = 1)
plt.xticks(rotation=45, ha = 'right')
plt.ylim((0,22000))
plt.xlabel(None)
plt.tight_layout()
plt.savefig(fn + '_ncounts.png')
plt.show()

In [ ]:
#extra make unique

In [12]:
dataframe.obs_names_make_unique()
dataframe.var_names_make_unique()

In [56]:
#subset dataframe (only if needed)

In [13]:
sam=SAM(dataframe)
sam.preprocess_data() #log transforms and filters the data
sam.run(batch_key='key') #run with default parameters

RUNNING SAM
Iteration: 0, Convergence: 1.0


2026-07-16 17:33:53,895 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-16 17:34:15,045 - harmonypy - INFO - sklearn.KMeans initialization complete.


Iteration: 1, Convergence: 0.8517164707443665


2026-07-16 17:36:35,770 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-16 17:36:58,718 - harmonypy - INFO - sklearn.KMeans initialization complete.


Iteration: 2, Convergence: 0.012213477511382919


2026-07-16 17:41:17,702 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-16 17:41:39,459 - harmonypy - INFO - sklearn.KMeans initialization complete.


Computing the UMAP embedding...
Elapsed time: 725.7005300521851 seconds


In [16]:
sam.adata

AnnData object with n_obs × n_vars = 66618 × 25503
    obs: 'n_genes', 'n_counts', 'key'
    var: 'mask_genes', 'means', 'variances', 'weights', 'spatial_dispersions'
    uns: 'preprocess_args', 'run_args', 'dimred_indices', 'ranked_genes'
    obsm: 'X_processed', 'X_pca', 'X_umap', 'Raw_X'
    varm: 'PCs'
    layers: 'X_disp'
    obsp: 'distances', 'connectivities', 'nnm'

In [ ]:
#save raw counts in obsm

In [17]:
sam.adata.obsm['Raw_X'] = dataframe.X

In [18]:
sam.save_anndata('../../Active_SAM_joined/SAM_DR_ncbi_07152026.h5ad')